# 01 — Geração dos cenários sintéticos

Cria o CCD, calibra deterministicamente as âncoras, diagnostica redundância, valida recuperação sem ruído e amostra diretamente o casco convexo. A referência não é descoberta por um otimizador.

In [ ]:
from pathlib import Path
import json, os, time, math, gc, hashlib
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.spatial import Delaunay
from scipy.stats import qmc

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper()
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='1'
ALPHA=2**0.75; DELTA_BY_K={2:.10,3:.10,4:.20,5:.50}
OUT=ROOT/'data'/'generated'; REF=ROOT/'data'/'reference_fronts'
if MODE=='SMOKE': OUT=OUT/'smoke'; REF=REF/'smoke'
elif MODE=='SCENARIO_AUDIT': OUT=OUT/'scenario_audit'; REF=REF/'scenario_audit'
OUT.mkdir(parents=True,exist_ok=True); REF.mkdir(parents=True,exist_ok=True)

def ccd3():
    from itertools import product
    factorial=np.array(list(product([-1.,1.],repeat=3)))
    axial=np.vstack([np.eye(3)*ALPHA,-np.eye(3)*ALPHA])
    return np.vstack([factorial,axial,np.zeros((5,3))])

def design(X):
    x1,x2,x3=np.asarray(X).T
    return np.column_stack([np.ones(len(x1)),x1,x2,x3,x1*x1,x2*x2,x3*x3,x1*x2,x1*x3,x2*x3])

X_CCD=ccd3(); Z=design(X_CCD)
assert X_CCD.shape==(19,3) and np.linalg.matrix_rank(Z)==10

def uniform_ball_sobol(n,seed):
    eng=qmc.Sobol(3,scramble=True,seed=seed); chunks=[]; total=0
    while total<n:
        u=eng.random_base2(15) if total==0 else eng.random(32768)
        y=2*u-1; y=y[np.einsum('ij,ij->i',y,y)<=1]
        chunks.append(y); total+=len(y)
    return np.vstack(chunks)[:n]*ALPHA

def base_directions(m, phase=0.):
    if m==4:
        u=np.array([[1,1,1],[1,-1,-1],[-1,1,-1],[-1,-1,1]],float)
        return u/np.linalg.norm(u,axis=1,keepdims=True)
    i=np.arange(m); z=1-2*(i+.5)/m; phi=np.pi*(1+np.sqrt(5))*(i+phase); r=np.sqrt(1-z*z)
    return np.column_stack([r*np.cos(phi),r*np.sin(phi),z])

def truth(X,A): return np.sum((np.asarray(X)[:,None,:]-A[None,:,:])**2,axis=2)
def rho_abs(F):
    C=np.corrcoef(F,rowvar=False)
    return float(np.mean(np.abs(C[np.triu_indices(C.shape[0],1)]))),C

X_CAL=uniform_ball_sobol(int(CFG['sobol_calibration_points']),2026)
X_SEARCH=X_CAL if len(X_CAL)<=8192 else X_CAL[:8192]

def decode_anchor_vector(v,m):
    raw=np.asarray(v).reshape(m,4); directions=raw[:,:3]
    directions/=np.maximum(np.linalg.norm(directions,axis=1,keepdims=True),1e-12)
    radii=.18*ALPHA+.70*ALPHA/(1+np.exp(-raw[:,3]))
    return directions*radii[:,None]

def anchor_penalties(A):
    distances=np.linalg.norm(A[:,None,:]-A[None,:,:],axis=2); distances+=np.eye(len(A))*1e6
    min_distance=float(distances.min()); singular=np.linalg.svd(A-A.mean(0),compute_uv=False)
    separation=max(0.,.10*ALPHA-min_distance)**2
    rank=max(0.,.025*ALPHA-singular[-1])**2
    return min_distance,singular,500*separation+500*rank

def calibrate_anchors(m,target,seed=7300):
    rng=np.random.default_rng(seed+100*m+round(100*target)); attempts=[]; best=None
    starts=int(CFG.get('anchor_search_starts',12)); maxiter=int(CFG.get('anchor_search_maxiter',350))
    for attempt in range(starts):
        directions=base_directions(m,attempt/starts) if attempt<3 else rng.normal(size=(m,3))
        directions/=np.linalg.norm(directions,axis=1,keepdims=True)
        radii=rng.uniform(.28,.78,size=m); v0=np.column_stack([directions,np.log(radii/(1-radii))]).ravel()
        def objective(v):
            A=decode_anchor_vector(v,m); rho,_=rho_abs(truth(X_SEARCH,A)); _,_,penalty=anchor_penalties(A)
            return (rho-target)**2+penalty
        opt=minimize(objective,v0,method='L-BFGS-B',options={'maxiter':maxiter,'ftol':1e-13,'maxls':30})
        A=decode_anchor_vector(opt.x,m); achieved,_=rho_abs(truth(X_CAL,A)); mind,singular,penalty=anchor_penalties(A)
        value=(achieved-target)**2+penalty
        row={'attempt':attempt,'optimizer_success':bool(opt.success),'optimizer_message':str(opt.message),'iterations':int(opt.nit),'objective':float(value),'achieved':achieved,'min_anchor_distance':mind,'affine_singular_values':singular.tolist()}
        attempts.append(row)
        if best is None or value<best[0]: best=(value,A,row)
        if abs(achieved-target)<=float(CFG['correlation_tolerance']) and mind>.05*ALPHA and singular[-1]>.01*ALPHA: break
    value,A,row=best
    if abs(row['achieved']-target)>float(CFG['correlation_tolerance']):
        raise RuntimeError(f'Calibração tecnicamente adequada falhou: m={m}, alvo={target}, melhor={row["achieved"]:.6f}')
    return A,attempts,row

rows=[]
for m in CFG['scenario_objectives']:
  for level,target in CFG['correlation_targets'].items():
    seed=7300+100*m+round(100*target); A,attempts,best=calibrate_anchors(m,float(target),seed)
    F=truth(X_CAL,A); rho,C=rho_abs(F); standardized=(F-F.mean(0))/F.std(0,ddof=1); s=np.linalg.svd(standardized,compute_uv=False); lam=s*s; p=lam/lam.sum(); reff=float(np.exp(-np.sum(p[p>0]*np.log(p[p>0]))))
    sid=f'm{m}_{level}'; anchor_hash=hashlib.sha256(np.ascontiguousarray(A).tobytes()).hexdigest()
    np.savez_compressed(OUT/f'{sid}_scenario.npz',anchors=A,correlation=C,singular_values=s,ideal_true=np.zeros(m),nadir_true=np.max(np.sum((A[:,None,:]-A[None,:,:])**2,axis=2),axis=0),anchor_hash=anchor_hash)
    (OUT/f'{sid}_anchor_search.json').write_text(json.dumps({'schema_version':2,'scenario':sid,'seed':seed,'objective':'(rho-target)^2 + separation/rank/feasibility penalties','attempt_count':len(attempts),'best':best,'attempts':attempts},indent=2),encoding='utf-8')
    rows.append({'scenario':sid,'m':m,'level':level,'target':target,'achieved':rho,'deviation':rho-target,'seed':seed,'search_attempts':len(attempts),'search_best_objective':best['objective'],'affine_rank':int(np.linalg.matrix_rank(A-A[0],tol=1e-10)),'effective_rank':reff,'redundancy':1-reff/m,'within_tolerance':abs(rho-target)<=CFG['correlation_tolerance'],'anchor_hash':anchor_hash})
scenarios=pd.DataFrame(rows); scenarios.to_csv(OUT/'scenario_diagnostics.csv',index=False)
assert len(scenarios)==len(CFG['scenario_objectives'])*len(CFG['correlation_targets']) and scenarios.within_tolerance.all()

def fit_rsm(A,seed,noisy=True):
    F=truth(X_CCD,A); rng=np.random.default_rng(seed); var=F.var(0,ddof=1); sigma=np.sqrt(var*(1-.95)/.95)
    Y=F+rng.normal(0,sigma,size=F.shape) if noisy else F
    B=np.linalg.lstsq(Z,Y,rcond=None)[0]; Yh=Z@B; E=Y-Yh; mse=np.sum(E*E,axis=0)/(len(Y)-Z.shape[1]); r2=1-np.sum(E*E,axis=0)/np.sum((Y-Y.mean(0))**2,axis=0)
    return B,mse,r2,sigma

for rec in rows:
    A=np.load(OUT/f"{rec['scenario']}_scenario.npz")['anchors']; B0,_,_,_=fit_rsm(A,101,False)
    probe=uniform_ball_sobol(1024,91); assert np.max(np.abs(design(probe)@B0-truth(probe,A)))<1e-10

def sample_convex_hull(A,n,seed):
    if n<len(A): raise ValueError('reference_points deve ser >= m para incluir todas as âncoras')
    tri=Delaunay(A); T=A[tri.simplices]; vols=np.abs(np.linalg.det(T[:,1:]-T[:,:1]))/6
    rng=np.random.default_rng(seed); remaining=n-len(A); ids=rng.choice(len(T),size=remaining,p=vols/vols.sum())
    e=rng.exponential(size=(remaining,4)); w=e/e.sum(1,keepdims=True); interior=np.einsum('ni,nij->nj',w,T[ids])
    X=np.vstack([A,interior]); assert np.max(np.linalg.norm(X,axis=1))<ALPHA
    return X

for rec in rows:
    A=np.load(OUT/f"{rec['scenario']}_scenario.npz")['anchors']; Xp=sample_convex_hull(A,int(CFG['reference_points']),404)
    ideal=np.zeros(len(A)); nadir=np.max(np.sum((A[:,None,:]-A[None,:,:])**2,axis=2),axis=0)
    np.savez_compressed(REF/f"{rec['scenario']}_pareto_reference.npz",X=Xp,F=truth(Xp,A),ideal_true=ideal,nadir_true=nadir,anchors_included=np.array(True))
print(scenarios[['scenario','target','achieved','within_tolerance','effective_rank']].to_string(index=False))


## Auditoria numérica dos cenários

Verifica explicitamente âncoras, ótimos, espectro, reprodutibilidade, casco e não dominância aproximada da referência.

In [ ]:
# EXPLICIT SCENARIO NUMERICAL AUDIT
audit=[]
for rec in rows:
    sid=rec['scenario']; bundle=np.load(OUT/f'{sid}_scenario.npz'); A=bundle['anchors']; m=len(A)
    pairwise=np.linalg.norm(A[:,None,:]-A[None,:,:],axis=2); pairwise[np.diag_indices(m)]=np.inf
    assert pairwise.min()>.05*ALPHA and np.max(np.linalg.norm(A,axis=1))<ALPHA and np.linalg.matrix_rank(A-A[0])==3
    anchor_values=truth(A,A); assert np.max(np.abs(np.diag(anchor_values)))<1e-12
    probe1=sample_convex_hull(A,256,5150+m); probe2=sample_convex_hull(A,256,5150+m); assert np.array_equal(probe1,probe2)
    assert np.all(Delaunay(A).find_simplex(probe1)>=0)
    ref=np.load(REF/f'{sid}_pareto_reference.npz'); assert np.array_equal(ref['X'][:m],A) and bool(ref['anchors_included'])
    ideal=np.zeros(m); nadir=np.max(np.sum((A[:,None,:]-A[None,:,:])**2,axis=2),axis=0); assert np.allclose(ref['ideal_true'],ideal) and np.allclose(ref['nadir_true'],nadir)
    Fref=ref['F']; rng=np.random.default_rng(8080+m); ids=rng.choice(len(Fref),size=min(512,len(Fref)),replace=False); Fs=Fref[ids]
    dominated=np.array([np.any(np.all(Fs<=f+1e-12,axis=1)&np.any(Fs<f-1e-10,axis=1)) for f in Fs]); assert dominated.mean()<.01
    Fcal=truth(X_CAL,A); standardized=(Fcal-Fcal.mean(0))/Fcal.std(0,ddof=1); singular=np.linalg.svd(standardized,compute_uv=False); variance=singular**2; cumulative=np.cumsum(variance)/variance.sum()
    audit.append({'scenario':sid,'min_anchor_distance':float(pairwise.min()),'max_anchor_norm':float(np.linalg.norm(A,axis=1).max()),'norms_json':json.dumps(np.linalg.norm(A,axis=1).tolist()),'numerical_rank':int(np.linalg.matrix_rank(standardized,tol=1e-10)),'singular_values_json':json.dumps(singular.tolist()),'cumulative_variance_json':json.dumps(cumulative.tolist()),'reference_reproducible':True,'reference_inside_hull':True,'reference_contains_all_anchors':True,'analytic_ideal_nadir_verified':True,'reference_approx_nondominated_fraction':float(1-dominated.mean()),'known_optima_verified':True})
audit=pd.DataFrame(audit); scenarios=scenarios.merge(audit,on='scenario',validate='one_to_one'); scenarios.to_csv(OUT/'scenario_diagnostics.csv',index=False)
assert scenarios.within_tolerance.all() and scenarios.affine_rank.eq(3).all()
print(f'Auditoria explícita aprovada para {len(audit)} cenários.')
